# RecData — Download & Extraction Showcase

This notebook demonstrates the `recdata` download and extraction pipeline.
It covers four scenarios you will encounter in practice:

| Scenario | Dataset | Situation |
|---|---|---|
| **A** | Steam / MovieLens-32M | Files already present locally — download skipped |
| **B** | Yelp | Files in a `.tar` archive on disk — selective extract only |
| **C** | MovieLens-latest-small | Files missing — download from URL, extract ZIP |
| **D** | xWines | Files missing, no source URL — clear error with instructions |

**Key design principle:** The pipeline always checks for local files first.
Download only happens when files are genuinely missing.

## Setup

In [7]:
import sys, json, tarfile, time
from pathlib import Path
import pandas as pd

# In a Jupyter notebook cwd() is the notebook directory.
# The repo root is one level up.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT    = NOTEBOOK_DIR.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATASETS_ROOT = Path("/Users/gethakloppers/Documents/A PhD/D Datasets")
RAW_DATA_ROOT = REPO_ROOT / "raw_data"
RAW_DATA_ROOT.mkdir(exist_ok=True)

print(f"Repo root      : {REPO_ROOT}")
print(f"Datasets root  : {DATASETS_ROOT}")
print(f"Raw data output: {RAW_DATA_ROOT}")

Repo root      : /Users/gethakloppers/Documents/A PhD/C GitRepos/Data-rec
Datasets root  : /Users/gethakloppers/Documents/A PhD/D Datasets
Raw data output: /Users/gethakloppers/Documents/A PhD/C GitRepos/Data-rec/raw_data


In [9]:
import logging
logging.basicConfig(level=logging.INFO,
                    format="%(levelname)s | %(name)s | %(message)s")

from recdata.loaders.base_loader import load_config
from recdata.loaders.downloader  import prepare_dataset, download_file, extract_archive
from recdata.exceptions          import DownloadError

print("All imports OK")

All imports OK


---
## How the config drives everything

Each dataset has a YAML config in `recdata/configs/`.
Two blocks are relevant to downloading:

- **`files`** — the filenames the pipeline expects to find in `raw_path`
- **`source`** — where to get the data if files are missing (URL, archive type, inner paths)

```yaml
source:
  url: "https://files.grouplens.org/datasets/movielens/ml-32m.zip"
  archive: zip
  checksum: null
  inner_paths:
    interactions: "ml-32m/ratings.csv"
    items:        "ml-32m/movies.csv"

files:
  interactions:
    filename: "ratings.csv"
    format:   csv
  items:
    filename: "movies.csv"
    format:   csv
```

`prepare_dataset(config, raw_path)` implements this logic:
```
for each file declared in config["files"]:
    if file exists locally (non-empty)  ->  return its path immediately
    elif source.url is set              ->  download archive, extract, clean up
    else                                ->  raise DownloadError with instructions
```

---
## Scenario A — Files already present locally

Steam and MovieLens-32M are already on disk.
`prepare_dataset` detects this and returns paths immediately — no network activity.

In [10]:
steam_config   = load_config(REPO_ROOT / "recdata/configs/steam.yaml")
steam_raw_path = DATASETS_ROOT / "steam"

print(f"Dataset : {steam_config['dataset_name']}")
print(f"Raw path: {steam_raw_path}\n")
print(f"{'Role':<14} {'Filename':<30} {'Format':<8} {'On disk?'}")
print("-" * 65)
for role, fdef in steam_config["files"].items():
    if fdef:
        p   = steam_raw_path / fdef["filename"]
        sz  = p.stat().st_size if p.exists() else 0
        ok  = f"✓  {sz/1e6:.0f} MB" if sz > 0 else "✗  missing"
        print(f"  {role:<12} {fdef['filename']:<30} {fdef['format']:<8} {ok}")

INFO | recdata.loaders.base_loader | Loaded config for dataset 'steam' from steam.yaml


Dataset : steam
Raw path: /Users/gethakloppers/Documents/A PhD/D Datasets/steam

Role           Filename                       Format   On disk?
-----------------------------------------------------------------
  interactions steam_reviews.json             jsonl    ✓  4273 MB
  items        steam_games.json               jsonl    ✓  21 MB


In [11]:
t0          = time.perf_counter()
steam_paths = prepare_dataset(steam_config, steam_raw_path)
elapsed     = time.perf_counter() - t0

print(f"prepare_dataset completed in {elapsed*1000:.1f} ms\n")
print("Resolved file paths:")
for role, path in steam_paths.items():
    print(f"  {role:<14} {path}")

INFO | recdata.loaders.downloader | Found local file for 'interactions': /Users/gethakloppers/Documents/A PhD/D Datasets/steam/steam_reviews.json
INFO | recdata.loaders.downloader | Found local file for 'items': /Users/gethakloppers/Documents/A PhD/D Datasets/steam/steam_games.json
INFO | recdata.loaders.downloader | All files found locally — skipping download


prepare_dataset completed in 7.8 ms

Resolved file paths:
  interactions   /Users/gethakloppers/Documents/A PhD/D Datasets/steam/steam_reviews.json
  items          /Users/gethakloppers/Documents/A PhD/D Datasets/steam/steam_games.json


In [12]:
ml_config   = load_config(REPO_ROOT / "recdata/configs/movielens32m.yaml")
ml_raw_path = DATASETS_ROOT / "ml-32m"

ml_paths = prepare_dataset(ml_config, ml_raw_path)

print("MovieLens-32M resolved paths:")
for role, path in ml_paths.items():
    print(f"  {role:<14} {path.name:<25} ({path.stat().st_size/1e6:,.0f} MB)")

INFO | recdata.loaders.base_loader | Loaded config for dataset 'movielens32m' from movielens32m.yaml
INFO | recdata.loaders.downloader | Found local file for 'interactions': /Users/gethakloppers/Documents/A PhD/D Datasets/ml-32m/ratings.csv
INFO | recdata.loaders.downloader | Found local file for 'items': /Users/gethakloppers/Documents/A PhD/D Datasets/ml-32m/movies.csv
INFO | recdata.loaders.downloader | Found local file for 'tags': /Users/gethakloppers/Documents/A PhD/D Datasets/ml-32m/tags.csv
INFO | recdata.loaders.downloader | All files found locally — skipping download


MovieLens-32M resolved paths:
  interactions   ratings.csv               (877 MB)
  items          movies.csv                (4 MB)
  tags           tags.csv                  (72 MB)


---
## Scenario B — Selective extraction from a TAR archive (Yelp)

The Yelp Academic Dataset ships as a single `yelp_dataset.tar`.
Because individual files are very large (the reviews JSON is 5.3 GB),
we extract **only the files we need** rather than unpacking everything.

We call `extract_archive` directly here so you can see each step.

In [13]:
yelp_tar = DATASETS_ROOT / "Yelp JSON" / "yelp_dataset.tar"

print(f"Archive : {yelp_tar}")
print(f"Size    : {yelp_tar.stat().st_size/1e9:.1f} GB\n")
print(f"{'Member':<55} {'Size (MB)':>10}")
print("-" * 67)
with tarfile.open(yelp_tar, "r") as tf:
    for m in sorted(tf.getmembers(), key=lambda m: m.size, reverse=True):
        if m.isfile():
            print(f"  {m.name:<53} {m.size/1e6:>9,.0f}")

Archive : /Users/gethakloppers/Documents/A PhD/D Datasets/Yelp JSON/yelp_dataset.tar
Size    : 4.3 GB

Member                                                   Size (MB)
-------------------------------------------------------------------
  yelp_academic_dataset_review.json                         5,342
  yelp_academic_dataset_user.json                           3,363
  yelp_academic_dataset_checkin.json                          287
  yelp_academic_dataset_tip.json                              181
  yelp_academic_dataset_business.json                         119
  Dataset_User_Agreement.pdf                                    0


In [15]:
# Extract only the business file (118 MB) for this demo.
# A full pipeline run would also pull out review + user.
yelp_dest     = RAW_DATA_ROOT / "yelp"
business_file = yelp_dest / "yelp_academic_dataset_business.json"

if business_file.exists() and business_file.stat().st_size > 0:
    print(f"Already extracted: {business_file.name}  "
          f"({business_file.stat().st_size/1e6:.0f} MB)")
else:
    print("Extracting business file from tar archive (118 MB)...")
    extracted = extract_archive(
        archive_path=yelp_tar,
        dest_path=yelp_dest,
        archive_type="tar",
        inner_paths={"items": "yelp_academic_dataset_business.json"},
    )
    for p in extracted:
        print(f"\n  Extracted: {p.name}  ({p.stat().st_size/1e6:.0f} MB)")

Already extracted: yelp_academic_dataset_business.json  (119 MB)


In [16]:
# Verify: peek at the first 5 records
samples = []
with open(business_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5: break
        samples.append(json.loads(line))

print(f"Columns: {list(samples[0].keys())}\n")
df_biz = pd.DataFrame(samples)[["name", "city", "state", "stars", "review_count", "categories"]]
print(df_biz.to_string(index=False))

Columns: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']

                    name          city state  stars  review_count                                                                                                 categories
Abby Rappoport, LAC, CMQ Santa Barbara    CA    5.0             7 Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, Acupuncture, Health & Medical, Nutritionists
           The UPS Store        Affton    MO    3.0            15                             Shipping Centers, Local Services, Notaries, Mailbox Centers, Printing Services
                  Target        Tucson    AZ    3.5            22                         Department Stores, Shopping, Fashion, Home & Garden, Electronics, Furniture Stores
      St Honore Pastries  Philadelphia    PA    4.0            80                                                      Restaurants, F

---
## Scenario C — Full download + extract from a URL

This demonstrates a complete download cycle using
[MovieLens-latest-small](https://grouplens.org/datasets/movielens/latest/)
— a ~1 MB public ZIP from GroupLens, which makes a fast and clean demo.

The same mechanism applies to any dataset whose config has `source.url` set.

In [17]:
# Inline config — in production this would be recdata/configs/movielens_small.yaml
demo_config = {
    "dataset_name": "movielens_small_demo",
    "files": {
        "interactions": {"filename": "ratings.csv", "format": "csv"},
        "items":        {"filename": "movies.csv",  "format": "csv"},
    },
    "schema": {
        "user_identifier": ["userId"],
        "item_identifier":  ["movieId"],
        "timestamp":        ["timestamp"],
        "rating":           ["rating"],
    },
    "source": {
        "url":     "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip",
        "archive": "zip",
        "checksum": None,
        "inner_paths": {
            "interactions": "ml-latest-small/ratings.csv",
            "items":        "ml-latest-small/movies.csv",
        },
    },
}

demo_dest = RAW_DATA_ROOT / "movielens_small_demo"
demo_dest.mkdir(exist_ok=True)

print("Files status before download:")
for role, fdef in demo_config["files"].items():
    p  = demo_dest / fdef["filename"]
    sz = p.stat().st_size if p.exists() else 0
    ok = f"✓  {sz/1e6:.2f} MB" if sz > 0 else "✗  not present"
    print(f"  {role:<14} {fdef['filename']:<20} {ok}")

Files status before download:
  interactions   ratings.csv          ✓  2.48 MB
  items          movies.csv           ✓  0.49 MB


In [18]:
# prepare_dataset:
#  1. Sees ratings.csv / movies.csv are missing
#  2. Downloads ml-latest-small.zip from GroupLens (~1 MB)
#  3. Extracts only the two declared inner paths
#  4. Deletes the zip archive
#  5. Returns { role -> Path }

t0         = time.perf_counter()
demo_paths = prepare_dataset(demo_config, demo_dest)
elapsed    = time.perf_counter() - t0

print(f"\nprepare_dataset finished in {elapsed:.1f} s\n")
print("Resolved paths:")
for role, path in demo_paths.items():
    print(f"  {role:<14} {path.name}  ({path.stat().st_size/1e6:.2f} MB)")

INFO | recdata.loaders.downloader | Found local file for 'interactions': /Users/gethakloppers/Documents/A PhD/C GitRepos/Data-rec/raw_data/movielens_small_demo/ratings.csv
INFO | recdata.loaders.downloader | Found local file for 'items': /Users/gethakloppers/Documents/A PhD/C GitRepos/Data-rec/raw_data/movielens_small_demo/movies.csv
INFO | recdata.loaders.downloader | All files found locally — skipping download



prepare_dataset finished in 0.0 s

Resolved paths:
  interactions   ratings.csv  (2.48 MB)
  items          movies.csv  (0.49 MB)


In [19]:
# Second call is instant — files exist, download skipped
t0 = time.perf_counter()
_ = prepare_dataset(demo_config, demo_dest)
print(f"Second call: {(time.perf_counter()-t0)*1000:.1f} ms "
      "— files found locally, download skipped ✓")

INFO | recdata.loaders.downloader | Found local file for 'interactions': /Users/gethakloppers/Documents/A PhD/C GitRepos/Data-rec/raw_data/movielens_small_demo/ratings.csv
INFO | recdata.loaders.downloader | Found local file for 'items': /Users/gethakloppers/Documents/A PhD/C GitRepos/Data-rec/raw_data/movielens_small_demo/movies.csv
INFO | recdata.loaders.downloader | All files found locally — skipping download


Second call: 5.3 ms — files found locally, download skipped ✓


In [20]:
# Verify the downloaded data
ratings = pd.read_csv(demo_paths["interactions"])
movies  = pd.read_csv(demo_paths["items"])

print(f"ratings.csv : {len(ratings):,} rows × {ratings.shape[1]} cols")
print(f"movies.csv  : {len(movies):,} rows × {movies.shape[1]} cols\n")

print("Ratings sample:")
print(ratings.head(3).to_string(index=False))
print("\nMovies sample:")
print(movies.head(3).to_string(index=False))

ratings.csv : 100,836 rows × 4 cols
movies.csv  : 9,742 rows × 3 cols

Ratings sample:
 userId  movieId  rating  timestamp
      1        1     4.0  964982703
      1        3     4.0  964981247
      1        6     4.0  964982224

Movies sample:
 movieId                   title                                      genres
       1        Toy Story (1995) Adventure|Animation|Children|Comedy|Fantasy
       2          Jumanji (1995)                  Adventure|Children|Fantasy
       3 Grumpier Old Men (1995)                              Comedy|Romance


---
## Scenario D — Missing files, no source URL (xWines)

The xWines CSV files on disk are empty (they need to be re-downloaded),
and the config has `source.url: null` because xWines requires
[manual download](https://github.com/rogerioxavier/X-Wines).

Rather than failing silently, the pipeline raises a `DownloadError`
explaining exactly which files are needed and where to place them.

In [23]:
xwines_config   = load_config(REPO_ROOT / "recdata/configs/xwines.yaml")
xwines_raw_path = DATASETS_ROOT / "xwines"

print("xWines files on disk:")
for role, fdef in xwines_config["files"].items():
    if fdef:
        p  = xwines_raw_path / fdef["filename"]
        sz = p.stat().st_size if p.exists() else 0
        ok = f"✓  {sz/1e6:.0f} MB" if sz > 0 else "✗  empty / missing"
        print(f"  {role:<14} {fdef['filename']:<35} {ok}")

print(f"\nsource.url : {xwines_config.get('source', {}).get('url')}")

INFO | recdata.loaders.base_loader | Loaded config for dataset 'xwines' from xwines.yaml


xWines files on disk:
  interactions   XWines_Full_21M_ratings.zip         ✓  311 MB
  items          XWines_Full_100K_wines.zip          ✓  5 MB

source.url : None


In [24]:
try:
    prepare_dataset(xwines_config, xwines_raw_path)
except DownloadError as e:
    print("DownloadError (expected):\n")
    print(str(e))

INFO | recdata.loaders.downloader | Found local file for 'interactions': /Users/gethakloppers/Documents/A PhD/D Datasets/xwines/XWines_Full_21M_ratings.zip
INFO | recdata.loaders.downloader | Found local file for 'items': /Users/gethakloppers/Documents/A PhD/D Datasets/xwines/XWines_Full_100K_wines.zip
INFO | recdata.loaders.downloader | All files found locally — skipping download


---
## Bonus — Using `download_file` directly

`download_file` can be used independently for one-off downloads
or when you want explicit checksum verification.

In [15]:
readme_dest = RAW_DATA_ROOT / "_demo" / "ml-small-README.html"
readme_dest.parent.mkdir(parents=True, exist_ok=True)
if readme_dest.exists(): readme_dest.unlink()  # always re-download to show progress

path = download_file(
    url="https://files.grouplens.org/datasets/movielens/ml-latest-small-README.html",
    dest_path=readme_dest,
    checksum=None,   # omit checksum for this demo
)

print(f"\nDownloaded : {path.name}  ({path.stat().st_size:,} bytes)")
print(f"\nContent preview (first 400 chars):")
print(path.read_text(encoding="utf-8")[:400])

INFO | recdata.loaders.downloader | Downloading: https://files.grouplens.org/datasets/movielens/ml-latest-small-README.html
INFO | recdata.loaders.downloader | Destination: /Users/gethakloppers/Documents/A PhD/C GitRepos/Data-rec/raw_data/_demo/ml-small-README.html
ml-small-README.html: 9.95kB [00:00, 9.14MB/s]                   
INFO | recdata.loaders.downloader | Downloaded 0.0 MB



Downloaded : ml-small-README.html  (9,947 bytes)

Content preview (first 400 chars):
<h1>Summary</h1>

<p>This dataset (ml-latest-small) describes 5-star rating and free-text tagging activity from <a href="http://movielens.org">MovieLens</a>, a movie recommendation service. It contains 100836 ratings and 3683 tag applications across 9742 movies. These data were created by 610 users between March 29, 1996 and September 24, 2018. This dataset was generated on September 26, 2018.</p>


---
## Summary

| Function | Purpose |
|---|---|
| `prepare_dataset(config, raw_path)` | One-call entrypoint — check local, download if missing, extract, return `{role: Path}` |
| `download_file(url, dest, checksum)` | Download a single file with optional MD5/SHA256 verification + progress bar |
| `extract_archive(archive, dest, type, inner_paths)` | Extract ZIP or TAR, optionally selecting only named inner files |

### Design decisions

| Decision | Rationale |
|---|---|
| **Skip-if-present** | Files are never re-downloaded if they exist and are non-empty |
| **Selective extraction** | Large archives (Yelp's 9 GB tar) only unpack the files that are needed |
| **Clear errors** | `DownloadError` tells you exactly which files are needed and where to put them |
| **`source` block is optional** | Works fine with manually-placed local data — no download machinery needed |
| **No index modification** | Downloaded files used as-is; ID remapping is a separate, explicit step |